In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

In [ ]:
# One-hot encode a DNA sequence
def one_hot_encode_sequence(sequence):
    base_to_idx = {'A': 0, 'C': 1, 'T': 2, 'G': 3}
    integer_encoded = [base_to_idx[base] for base in sequence]
    one_hot_encoded = np.eye(len(base_to_idx))[integer_encoded]  # shape: (sequence_length, 4)
    return one_hot_encoded

# Load dataset and apply one-hot encoding to all sequences
def prepare_data(input_file_path, sol):
    df = pd.read_csv(input_file_path, low_memory=False)
    df = df.dropna(subset=[f'{sol}_seq', f'{sol}_FRET'])  # remove rows with missing values

    X = df[f'{sol}_seq']
    y = df[f'{sol}_FRET']
    
    one_hot_encoded_X = [one_hot_encode_sequence(seq) for seq in X]

    return np.array(one_hot_encoded_X), y

# Build a simple RNN model based on sequence shape
def compile_model_RNN(X_train, activation='sigmoid', optimizer='adam', loss='mae', metrics=['mae']):
    batch_size, time_steps, features = X_train.shape

    model = Sequential()
    model.add(SimpleRNN(32, activation=activation, input_shape=(time_steps, features)))
    model.add(Dense(1, activation='linear'))

    model.compile(optimizer=optimizer, loss=loss, metrics=metrics)
    return model

# Train the model and plot training/validation loss
def train_model(model, X_train, y_train, val_size, batch_size=64, epochs=30):
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=val_size, random_state=42)

    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size,
                        validation_data=(X_val, y_val))

    fig = plt.figure()
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend(loc='upper right')
    plt.grid(True)
    plt.show()

    return model, fig

# Test the model and plot absolute error
def test_model(model, X_test, y_test):
    y_pred = model.predict(X_test).flatten()
    y_diff = y_test - y_pred
    y_abs = np.abs(y_diff)

    fig = plt.figure()
    plt.plot(y_abs, label='|test - pred|')
    plt.legend()

    test_loss = mean_absolute_error(y_test, y_pred)
    print(f"test_loss: {test_loss:.4f}")
    return fig

# Predict FRET for a new sequence
def predict_FRET(model, new_seq):
    new_seq_encoded = one_hot_encode_sequence(new_seq)
    predicted_FRET = model.predict(np.expand_dims(new_seq_encoded, axis=0))  # shape: (1, seq_len, 4)
    return predicted_FRET

# Pad one-hot encoded sequences to the same length
def pad_sequences(sequences, max_length):
    padded_sequences = []
    for seq in sequences:
        padding = np.zeros((max_length - len(seq), seq.shape[1]))
        padded_seq = np.vstack([seq, padding])
        padded_sequences.append(padded_seq)
    return np.array(padded_sequences)

In [ ]:
# Sequence types to run the model on
solution = ['N5', 'N50', 'N500', 'N5M10', 'N5M100']

# Hyperparameters
units = 32
batch_size = 128
epochs = 300
max_length = 9  # pad all sequences to this length

# Output directory (modify as needed)
output_directory_path = 'OUTPUT_DIRECTORY_PATH'

for sol in solution:
    print(sol)

    # Define input/output paths
    input_file_path = 'INPUT_FILE_PATH'
    output_file_path_model = f'{output_directory_path}/rnn/model/{sol}_un{units}_ep{epochs}_bs{batch_size}_RNN_padded.keras'
    output_file_path_fig_train = f'{output_directory_path}/rnn/fig_train/{sol}_un{units}_ep{epochs}_bs{batch_size}_train_RNN_padded.png'
    output_file_path_fig_test = f'{output_directory_path}/rnn/fig_test/{sol}_un{units}_ep{epochs}_bs{batch_size}_test_RNN_padded.png'

    # Prepare X and y (one-hot encoded sequence and FRET)
    X, y = prepare_data(input_file_path, sol)

    # Pad sequences to uniform length
    X = pad_sequences(X, max_length)
    print(X[0])  # check shape

    # Split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Compile RNN model
    model = compile_model_RNN(X_train)

    # Train model and plot training loss
    model, fig_train = train_model(model, X_train, y_train, val_size=0.15, batch_size=batch_size, epochs=epochs)

    # Evaluate model and plot prediction error
    fig_test = test_model(model, X_test, y_test)

    # Save model and figures
    model.save(output_file_path_model)
    fig_train.savefig(output_file_path_fig_train)
    fig_test.savefig(output_file_path_fig_test)